# Class 5 - Geospatial Data (netCDF & xarray)

Recommended documentation: [Geospatial course from UGhent](https://github.com/plovercode/DS-python-geospatial/tree/main/notebooks)


Today we move from **time-series tabular data** (pandas, Class 4) to **multi-dimensional gridded data**.

Goals:
- understand what a netCDF file is and how to read it with  **xarray** ,
- select, slice, reduce, filter, `groupby`, and `resample` gridded temperature data,
- apply simple raster operations: value masking and selecting a geographic region with **regionmask**.

> **Scope of this course:** we only work with **raster / netCDF** data. We will **not** cover shapefiles or vector GIS (GeoPandas, etc.).

## 1. From pandas to xarray

In Class 4, each row was one time step and each column was one variable (temperature, humidity, …). That works well for a **point station**.

Gridded climate data has more axes at once, for example:

- **time** (hourly steps in January 2024),
- **latitude**,
- **longitude**,
- and one or more **data variables** (here: predicted temperature).

![xarray Dataset: data variables, coordinates, and attributes](img/xarray-dataset.png)

*Figure: an `xarray.Dataset` can hold several data variables that share coordinates (e.g. lon, lat, time), plus metadata in attributes. (Source: [xarray documentation](https://docs.xarray.dev/).)*


## 2. The data

We use Future Forests **predicted 2 m air temperature** (`pred_temperature_C`) for January 2024 on a ~100 m grid over the project area provided by Christopher Jung.

The raw file is stored in projected coordinates (EPSG:25832, ETRS89 / UTM zone 32N). For class, we use a **preprocessed lon/lat (WGS84)** version:

```text
../processed_data/pred_temperature_C_2024_01_wgs84.nc
```

In [1]:
# Running this cell can take a few minutes
import matplotlib.pyplot as plt
import xarray as xr
import regionmask

## 3. Opening a netCDF with xarray

`xr.open_dataset` reads the file and returns a **Dataset**: a dictionary-like container of labelled arrays (DataArrays) that share dimensions.


In [2]:
DataPath = "../processed_data/pred_temperature_C_2024_01_wgs84.nc"
# Lazy open: do not load the full ~3 GB grid into memory at once
TempDs = xr.open_dataset(DataPath, chunks="auto")
TempDs

<xarray.Dataset> Size: 11GB
Dimensions:             (time: 744, lat: 2132, lon: 1701)
Coordinates:
  * time                (time) datetime64[ns] 6kB 2024-01-01 ... 2024-01-31T2...
  * lat                 (lat) float64 17kB 49.65 49.65 49.65 ... 47.48 47.48
  * lon                 (lon) float64 14kB 7.311 7.312 7.313 ... 9.042 9.043
Data variables:
    pred_temperature_C  (time, lat, lon) float32 11GB dask.array<chunksize=(166, 474, 378), meta=np.ndarray>
    crs                 int64 8B ...
Attributes: (12/13)
    Conventions:            CF-1.8
    title:                  Predicted temperature on 100 m grid (reprojected ...
    source_file:            ["D:\\ERA5\\Future_Forests_Area\\d2m_C\\ERA5_2024...
    model_features:         ["Albedo_Albedo", "Albedo_Multiscale_Features_Alb...
    baseline_feature:       t2m_C
    baseline_scale_factor:  0.1
    ...                     ...
    institution:            Future Forests
    references:             LightGBM model application
    comment:                Prediction was calculated only for cells where th...
    featureType:            grid
    variable:               pred_temperature_C
    history:                Created on 2026-07-30T11:34:52.391813; Reprojecte...

On the table above we can observe:
1. There are 3 dimensions which are the 3 coordinates; `time` of size 744 (why?), `lat` of size 2132 and `lon` of size 1701.
2. There are two data variables, but only one is usuable (`crs` is for metadata): `pred_temperature_C` of shape `(time, lon, lat)`

Let's select the variable that interests us (there is only one variable in this file but there could be multiple)

In [3]:
Temp = TempDs["pred_temperature_C"]
print("Variable attrs:", Temp.attrs)

Variable attrs: {'standard_name': 'air_temperature', 'long_name': 'predicted 2 metre air temperature', 'units': 'degree_Celsius', 'grid_mapping': 'crs'}


## 4. Selecting and slicing data

Two ways to index:

- **`.isel(...)`** — by **integer position** (like NumPy),
- **`.sel(...)`** — by **coordinate label** (like pandas `.loc`).

In [4]:
# Position-based: first time step
FirstMap = Temp.isel(time=0)

Quick map of the first time step with matplotlib. For a **2D grid** (lon × lat), use `plt.pcolormesh` to colour each cell, not `plt.plot` which draws lines (we use that for time series below).


In [ ]:
plt.pcolormesh(FirstMap.lon, FirstMap.lat, FirstMap, cmap="coolwarm", shading="auto")
plt.colorbar(label="Temperature (°C)")
plt.title("Predicted 2 m temperature at t=0")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
# Label-based: temperature near Hartheim station (~7.60°E, 47.93°N)
HartheimLon, HartheimLat = 7.60, 47.93
HartheimSeries = Temp.sel(lon=HartheimLon, lat=HartheimLat, method="nearest") # Nearest is used here for inexact matches

In [ ]:
plt.figure(figsize=(10, 3))
plt.plot(HartheimSeries.time, HartheimSeries)
plt.ylabel("Temperature (°C)")
plt.title("Hartheim, hourly predicted temperature, January 2024")
plt.show()


Spatial and temporal **slices** use `slice`


In [ ]:
# Small box around Freiburg / Hartheim
FreiburgBox = Temp.sel(
    lon=slice(7.5, 8.1),
    lat=slice(48.15, 47.75),  # high → low if lat decreases with index
    time=slice("2024-01-10", "2024-01-12"),
)
print(FreiburgBox.sizes)

BoxMap = FreiburgBox.isel(time=0)

plt.pcolormesh(BoxMap.lon, BoxMap.lat, BoxMap, cmap="coolwarm", shading="auto")
plt.colorbar(label="Temperature (°C)")
plt.title("Freiburg–Hartheim box — one timestep")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

> **Note:** for latitude, values decrease north to south, this is not always the case.


## 5. Reduction (collapse dimensions)

Let's say you want a map of average January temperature. You have hourly values at every grid cell, so you need to collapse the time dimension and keep the spatial ones.

Reductions such as `.mean()`, `.min()`, `.max()` can run along one or more named dimensions. It is the same idea as pandas aggregations, but you name the axis explicitly.


In [ ]:
JanMeanMap = Temp.mean(dim="time")

plt.pcolormesh(JanMeanMap.lon, JanMeanMap.lat, JanMeanMap, cmap="coolwarm", shading="auto")
plt.colorbar(label="Temperature (°C)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


Now ask how the region as a whole evolved during January. This time you collapse lat and lon and keep time, which leaves you with a single time series instead of a map.

In [ ]:
# Spatial mean over the Freiburg box → one value per time
BoxSpatialMean = FreiburgBox.mean(dim=("lat", "lon"))

plt.plot(BoxSpatialMean)
plt.ylabel("Temperature (°C)")
plt.title("Freiburg–Hartheim box — spatial mean")
plt.show()

## 6. Filtering / masking with `.where`

`.where(condition)` keeps values where the condition is True and sets the rest to `NaN` (grid shape unchanged — useful for maps).


In [ ]:
WarmMap = Temp.sel(lon=slice(7.5, 8.1), lat=slice(48.15, 47.75)).isel(time=12)
WarmCells = WarmMap.where(WarmMap > 5)

plt.figure(figsize=(8, 6))
plt.pcolormesh(WarmCells.lon, WarmCells.lat, WarmCells, cmap="YlOrRd", shading="auto")
plt.colorbar(label="Temperature (°C)")
plt.title("Cells warmer than 5 °C (example timestep, Freiburg–Hartheim box)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


## 7. `groupby` and `resample`

Like pandas, xarray supports split–apply–combine.

- **`groupby("time.hour")`** — group by clock hour → **diurnal cycle**
- **`resample(time="1D")`** — regular time bins (daily means)
- **`resample(time="YE")`** — yearly bins (with only January 2024 you get a single period; useful API for longer archives)


In [ ]:
# Diurnal cycle on a Freiburg–Hartheim box (faster than the full domain)
DemoBox = Temp.sel(lon=slice(7.5, 8.1), lat=slice(48.15, 47.75))
BoxSeries = DemoBox.mean(dim=("lat", "lon"))
Diurnal = BoxSeries.groupby("time.hour").mean()

plt.figure(figsize=(8, 3))
plt.plot(Diurnal.hour, Diurnal, marker="o")
plt.xlabel("Hour of day (UTC)")
plt.ylabel("Temperature (°C)")
plt.title("Freiburg–Hartheim box — diurnal cycle, January 2024")
plt.show()


In [ ]:
# Daily means
Daily = BoxSeries.resample(time="1D").mean()

plt.figure(figsize=(10, 3))
plt.plot(Daily.time, Daily, marker="o")
plt.ylabel("Temperature (°C)")
plt.title("Freiburg–Hartheim box — daily mean temperature, January 2024")
plt.show()


In [ ]:
# Yearly resample API (one group only with this file)
Yearly = BoxSeries.resample(time="YE").mean()
print(Yearly)


## Exercise A

A colleague asks for the January 2024 temperature time series at the **Hartheim** site for comparison with the station logger from Class 4.

Tasks:
1. Select `pred_temperature_C` at lon `7.60`, lat `47.93` using nearest-neighbour `.sel`.
2. Plot the hourly series for the whole month.
3. Print the mean and minimum temperature at that point.


In [ ]:
# Your code here


In [ ]:
# Solution
HartheimLon, HartheimLat = 7.60, 47.93
HartheimTs = Temp.sel(lon=HartheimLon, lat=HartheimLat, method="nearest")

plt.figure(figsize=(10, 3))
plt.plot(HartheimTs.time, HartheimTs)
plt.ylabel("Temperature (°C)")
plt.title("Hartheim — predicted 2 m temperature, January 2024")
plt.show()

print(f"Mean: {float(HartheimTs.mean()):.2f} °C")
print(f"Min:  {float(HartheimTs.min()):.2f} °C")


## Exercise B

You want a January overview map and the coldest day in a Freiburg analysis box.

Tasks:
1. Select the box `lon=7.5–8.1`, `lat=48.15→47.75` (north→south order matches this file).
2. Compute the **time-mean** map for that box and plot it.
3. For the same box, compute the spatial mean at each time, find the **coldest** timestep, and print its timestamp and temperature.


In [ ]:
# Your code here


In [ ]:
# Solution
Box = Temp.sel(lon=slice(7.5, 8.1), lat=slice(48.15, 47.75))
JanMean = Box.mean(dim="time").compute()

plt.figure(figsize=(8, 6))
plt.pcolormesh(JanMean.lon, JanMean.lat, JanMean, cmap="coolwarm", shading="auto")
plt.colorbar(label="Temperature (°C)")
plt.title("January mean predicted temperature (Freiburg–Hartheim box)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

BoxMeanTs = Box.mean(dim=("lat", "lon")).compute()
ColdIdx = BoxMeanTs.argmin(dim="time")
ColdTime = BoxMeanTs.time.isel(time=ColdIdx)
ColdValue = BoxMeanTs.isel(time=ColdIdx)
print("Coldest time:", str(ColdTime.values)[:19])
print(f"Coldest box-mean temperature: {float(ColdValue):.2f} °C")


## Exercise C

Forest microclimate often shows a clear day–night contrast. Using the **Hartheim** point series from Exercise A:

Tasks:
1. Build the **diurnal cycle** with `groupby("time.hour").mean()` and plot it.
2. Resample the same series to **daily** means (`resample(time="1D").mean()`) and plot it.


In [ ]:
# Your code here


In [ ]:
# Solution
HartheimTs = Temp.sel(lon=7.60, lat=47.93, method="nearest")

HartheimDiurnal = HartheimTs.groupby("time.hour").mean()
plt.figure(figsize=(8, 3))
plt.plot(HartheimDiurnal.hour, HartheimDiurnal, marker="o")
plt.xlabel("Hour of day (UTC)")
plt.ylabel("Temperature (°C)")
plt.title("Hartheim diurnal cycle — January 2024")
plt.show()

HartheimDaily = HartheimTs.resample(time="1D").mean()
plt.figure(figsize=(10, 3))
plt.plot(HartheimDaily.time, HartheimDaily, marker="o")
plt.ylabel("Temperature (°C)")
plt.title("Hartheim daily mean temperature — January 2024")
plt.show()


## 8. Raster operations: masking and regions

Two common raster tasks:

1. **Value masking** — keep only pixels that meet a condition (`.where`).
2. **Geographic regions** — keep only pixels inside a polygon.

We use **`regionmask`** with **manually defined lon/lat polygons** (no shapefiles).


### i. Value masking (demo)


In [ ]:
Day = Temp.sel(lon=slice(7.5, 8.1), lat=slice(48.15, 47.75)).sel(
    time="2024-01-15T12:00", method="nearest"
)
AboveZero = Day.where(Day > 0)

plt.figure(figsize=(8, 6))
plt.pcolormesh(AboveZero.lon, AboveZero.lat, AboveZero, cmap="YlOrRd", shading="auto")
plt.colorbar(label="Temperature (°C)")
plt.title("15 Jan ~12:00 — cells above 0 °C (Freiburg–Hartheim box)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


### ii. Bounding box vs polygon

A lon/lat **slice** is a rectangle. For irregular regions (valley vs foothills), define polygons and build a mask with `regionmask.Regions`.


In [ ]:
# Work on a regional extract for speed (full file is multi-GB)
RegionTemp = Temp.sel(lon=slice(7.4, 8.3), lat=slice(48.20, 47.70))

# Two simple teaching regions (lon/lat corners, closed rings)
UpperRhineVerts = [
    [7.45, 47.85],
    [7.70, 47.85],
    [7.70, 48.10],
    [7.45, 48.10],
    [7.45, 47.85],
]
BlackForestVerts = [
    [7.85, 47.80],
    [8.20, 47.80],
    [8.20, 48.05],
    [7.85, 48.05],
    [7.85, 47.80],
]

Regions = regionmask.Regions(
    [UpperRhineVerts, BlackForestVerts],
    names=["Upper Rhine", "Black Forest foothills"],
    abbrevs=["UR", "BF"],
    name="Future Forests teaching regions",
)
Regions


In [ ]:
# Mask aligned to the regional temperature grid (lon/lat)
Mask = Regions.mask(RegionTemp.lon, RegionTemp.lat)

plt.figure(figsize=(8, 6))
plt.pcolormesh(Mask.lon, Mask.lat, Mask, shading="auto")
plt.colorbar(label="Region id")
plt.title("Region mask (integer region id; NaN = outside)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


In [ ]:
# Mean January temperature inside each region
JanMean = RegionTemp.mean(dim="time").compute()
for RegionId, RegionName in enumerate(Regions.names):
    Regional = JanMean.where(Mask == RegionId)
    MeanVal = float(Regional.mean())
    print(f"{RegionName}: {MeanVal:.2f} °C")


In [ ]:
# Time series of spatial mean in the Upper Rhine region
UpperRhineTs = RegionTemp.where(Mask == 0).mean(dim=("lat", "lon")).compute()
BlackForestTs = RegionTemp.where(Mask == 1).mean(dim=("lat", "lon")).compute()

UpperRhineDaily = UpperRhineTs.resample(time="1D").mean()
BlackForestDaily = BlackForestTs.resample(time="1D").mean()

plt.figure(figsize=(10, 3))
plt.plot(UpperRhineDaily.time, UpperRhineDaily, label="Upper Rhine")
plt.plot(BlackForestDaily.time, BlackForestDaily, label="Black Forest foothills")
plt.legend()
plt.ylabel("Temperature (°C)")
plt.title("Daily mean temperature by region")
plt.show()


## Exercise D

On a frosty morning, you want to see where the predicted temperature stays below freezing near Freiburg.

Tasks:
1. Select the Freiburg–Hartheim box (`lon=7.5–8.1`, `lat=48.15→47.75`), then the map for `2024-01-20T06:00` (nearest).
2. Mask so that **only cells < 0 °C** remain (others `NaN`).
3. Plot the masked map.


In [ ]:
# Your code here


In [ ]:
# Solution
FrostMorning = Temp.sel(lon=slice(7.5, 8.1), lat=slice(48.15, 47.75)).sel(
    time="2024-01-20T06:00", method="nearest"
)
BelowZero = FrostMorning.where(FrostMorning < 0)

plt.figure(figsize=(8, 6))
plt.pcolormesh(BelowZero.lon, BelowZero.lat, BelowZero, cmap="Blues_r", shading="auto")
plt.colorbar(label="Temperature (°C)")
plt.title("20 Jan ~06:00 — cells below 0 °C (Freiburg–Hartheim box)")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()


## Exercise E

Define a region covering the **Hartheim–Freiburg** corridor and summarise January inside it.

Tasks:
1. Create a `regionmask.Regions` polygon roughly covering lon `7.55–7.95`, lat `47.85–48.05`.
2. Build a mask on `Temp.lon` / `Temp.lat`.
3. Compute the **daily mean** temperature inside that region for January and plot it.


In [ ]:
# Your code here


In [ ]:
# Solution
HartheimFreiburgVerts = [
    [7.55, 47.85],
    [7.95, 47.85],
    [7.95, 48.05],
    [7.55, 48.05],
    [7.55, 47.85],
]
HfRegions = regionmask.Regions(
    [HartheimFreiburgVerts],
    names=["Hartheim–Freiburg"],
    abbrevs=["HF"],
    name="Hartheim–Freiburg corridor",
)
HfTemp = Temp.sel(lon=slice(7.4, 8.1), lat=slice(48.15, 47.75))
HfMask = HfRegions.mask(HfTemp.lon, HfTemp.lat)

HfDaily = (
    HfTemp.where(HfMask == 0)
    .mean(dim=("lat", "lon"))
    .resample(time="1D")
    .mean()
    .compute()
)

plt.figure(figsize=(10, 3))
plt.plot(HfDaily.time, HfDaily, marker="o")
plt.ylabel("Temperature (°C)")
plt.title("Hartheim–Freiburg region — daily mean temperature, January 2024")
plt.show()


## 9. Wrap-up

- **netCDF + xarray** = labelled multi-dimensional arrays (Dataset / DataArray).
- Prefer **`.sel`** with coordinates; use **`.isel`** for positions.
- **Reduce** with `.mean("time")` / `.mean(("lat", "lon"))`; **filter** with `.where`.
- **`groupby`** / **`resample`** work much like pandas, on N-D data.
- **regionmask** selects geographic regions from lon/lat polygons without teaching shapefiles.

Next steps when you have longer archives: the same `resample(time="YE")` / seasonal `groupby("time.season")` patterns scale up unchanged.
